# GP - evolved flow-control programs

Target T2 (first failures only), project `atlasdb`. Features are the frozen list from `feature_selection.ipynb`; N = statement budget is the feature count.

Two fitness variants: SO (MMC minus a size penalty, tournament selection) and MO (multi-objective, TPR vs. FPR, NSGA-II selection).

In [ ]:
import pathlib
import time
from functools import partial

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from acise_p1_cigp.analysis.features import load_features
from acise_p1_cigp.data.load import load_project, preprocess
from acise_p1_cigp.gp.engine import GPConfig, evolve, fit_predict, statements, predict
from acise_p1_cigp.gp.simplify import motifs, to_python
from acise_p1_cigp.metrics.metrics import majority, parrot, scores
from acise_p1_cigp.stats.tests import summary
from acise_p1_cigp.training.cv import dev_split, folds, prepare, run_cv, Fold
from acise_p1_cigp.utils.logging.log import get_logger, set_level
from acise_p1_cigp.viz import plots

pio.renderers.default = "notebook+png"
logger = get_logger("notebook.gp")

TARGET, PROJECT, SEEDS = "T2", "atlasdb", range(10)
OUT = pathlib.Path("../results/gp").resolve()
OUT.mkdir(parents=True, exist_ok=True)

## Project loading...

In [ ]:
ds = preprocess(load_project(PROJECT), TARGET)
features = load_features(TARGET, PROJECT)
logger.info(f"Loaded {len(ds.X)} builds, {int(ds.y.sum())} failures, {len(features)} features: {features}.")

In [ ]:
dev = dev_split(ds, features)
for v in ("so", "mo"):
    t0 = time.perf_counter()
    r = evolve(dev.X_fit, dev.y_fit, dev.X_val, dev.y_val, GPConfig(variant=v), seed=0)
    t = time.perf_counter() - t0

    logger.info(f"{v.upper()}: {t:.2f}s, val MCC: {r.val['mcc']:.4f}")

## GP tuning on dev split

Small grid search on the dev split, for demo purposes.

In [ ]:
set_level("WARNING")
TUNE_SEEDS = range(3)
grid = [("so", dict(pop=p, tournament_n=t, lambda_=l)) for p in (200, 400) for t in (3, 5) for l in (0.02, 0.05, 0.1)]
grid += [("mo", dict(pop=p)) for p in (200, 300, 400, 500)]
rows = []
for v, params in grid:
    for s in TUNE_SEEDS:
        t0 = time.perf_counter()
        r = evolve(dev.X_fit, dev.y_fit, dev.X_val, dev.y_val, GPConfig(variant=v, **params), s)
        rows.append(
            {
                "variant": v,
                **params,
                "seed": s,
                "val_mcc": r.val["mcc"],
                "statements": statements(r.best),
                "time": time.perf_counter() - t0,
            }
        )
set_level("INFO")
tuning = pd.DataFrame(rows)
tuning.to_csv(OUT / f"tuning_{TARGET}_{PROJECT}.csv", index=False)
keys = ["variant", "pop", "tournament_n", "lambda_"]
(
    tuning.groupby(keys, dropna=False)[["val_mcc", "statements", "time"]]
    .agg(["median", "std"])
    .round(3)
    .sort_values(("val_mcc", "median"), ascending=False)
    .groupby(level="variant")
    .head(3)
)

In [ ]:
display(tuning.describe(include="all").style.format({"val_mcc": "{:.4f}", "time": "{:.2f}"}))
display(
    tuning.groupby(["variant", "pop", "tournament_n", "lambda_"])[["val_mcc", "statements"]]
    .median()
    .sort_values("val_mcc", ascending=False)
    .groupby("variant")
    .head(5)
)

## Actual training

No best performers above; we will use the best configuration in SO and MO settings.

In [ ]:
CFG = {
    "so": GPConfig(variant="so", pop=200, tournament_n=3, lambda_=0.01),
    "mo": GPConfig(variant="mo", pop=400, tournament_n=3, lambda_=0.05),
}

In [ ]:
set_level("WARNING")
fold = folds(ds)[-1]
rows = []
for frac in (0.25, 0.5, 0.75, 1.0):
    fit = fold.fit[-int(len(fold.fit) * frac):]
    sp = prepare(ds, features, Fold(fold.k, fit, fold.val, fold.test))
    for s in range(5):
        r = evolve(sp.X_fit, sp.y_fit, sp.X_val, sp.y_val, CFG["so"], s)
        test = scores(ds.y.iloc[fold.test], predict(r.best, r.pset, sp.X_test))
        rows += [{"fit_rows": len(fit), "seed": s, "split": "val", "mcc": r.val["mcc"], "program": str(r.best)},
                 {"fit_rows": len(fit), "seed": s, "split": "test", "mcc": test["mcc"], "program": str(r.best)}]
set_level("INFO")
lc = pd.DataFrame(rows)
lc.to_csv(OUT / f"learning_curve_{TARGET}_{PROJECT}.csv", index=False)
plots.save(plots.learning_curve(lc), f"gp_learning_curve_{TARGET}_{PROJECT}").show()
lc.groupby(["split", "fit_rows"])["mcc"].agg(["median", "min", "max"]).round(3)

In [ ]:
frames, payloads = [], {}
for v, cfg in CFG.items():
    df, payloads[v] = run_cv(ds, features, partial(fit_predict, cfg=cfg), f"gp_{v}", seeds=SEEDS)
    frames.append(df)
for name, fn in ("parrot", parrot), ("majority", majority):
    frames.append(run_cv(ds, features, fn, name)[0])
results = pd.concat(frames, ignore_index=True)
results.to_csv(OUT / f"cv_{TARGET}_{PROJECT}.csv", index=False)
summary(results)

## Convergence

Best line flat over the last half: fewer generations would do. Mean far below best: diverse population, or stuck on poor programs.

In [ ]:
for v, pl in payloads.items():
    stats = pd.concat([r.stats.assign(fold=k) for (k, _), r in pl.items()])
    stats.to_csv(OUT / f"stats_{v}_{TARGET}_{PROJECT}.csv", index=False)
    plots.save(plots.convergence(stats), f"gp_convergence_{v}_{TARGET}_{PROJECT}").show()

## Programs

Best program per run, pciked on validation fold. Shown simplified, in raw units of the last fold's scaler.

In [ ]:
programs = pd.DataFrame(
    [
        {
            "variant": v,
            "fold": k,
            "seed": s,
            "program": str(r.best),
            "statements": statements(r.best),
            "val_mcc": r.val["mcc"],
        }
        for v, pl in payloads.items()
        for (k, s), r in pl.items()
    ]
)
programs.to_csv(OUT / f"programs_{TARGET}_{PROJECT}.csv", index=False)

k = max(f.k for f in folds(ds))
prep = prepare(ds, features, folds(ds)[k]).prep
for v, pl in payloads.items():
    r = max((r for (f, _), r in pl.items() if f == k), key=lambda r: r.val["mcc"])
    display(Markdown(f"```python\n{to_python(r.best, r.pset, prep, name=f'gp_{v}_fold{k}')}```"))

## Pareto front

MO, last fold, see 0. Chosen program is closest to the upper left corner (max TPR, min FPR).

In [ ]:
r = payloads["mo"][(k, 0)]
chosen = r.table.loc[r.table["program"] == str(r.best)].iloc[0]
plots.save(plots.pareto(r.table, chosen), f"gp_pareto_{TARGET}_{PROJECT}").show()

## Program size vs. fitness (SO)

In [ ]:
hof = pd.concat([r.table.assign(fold=k, seed=s) for (k, s), r in payloads["so"].items()])
plots.save(plots.size_vs_fitness(hof), f"gp_size_fitness_{TARGET}_{PROJECT}").show()

## Feature frequency in HoF

In [ ]:
m = motifs([t for r in payloads["so"].values() for t in r.hof])
m.to_csv(OUT / f"motifs_{TARGET}_{PROJECT}.csv", index=False)
plots.save(plots.feature_freq_performers(m.groupby("feature")["count"].sum()),
           f"gp_feature_freq_{TARGET}_{PROJECT}").show()
m.head(15)

## Dev-segment runs for program (b)

Used for distilling the flow-control program from the GP's results.

In [ ]:
dev_runs = [evolve(dev.X_fit, dev.y_fit, dev.X_val, dev.y_val, CFG["so"], s) for s in SEEDS]
cand = pd.concat([r.table.assign(seed=r.seed) for r in dev_runs]).sort_values("val_fitness", ascending=False)
cand.to_csv(OUT / f"dev_candidates_{TARGET}_{PROJECT}.csv", index=False)
best_dev = max(dev_runs, key=lambda r: r.val["mcc"])
display(Markdown(f"```python\n{to_python(best_dev.best, best_dev.pset, dev.prep)}```"))
motifs([t for r in dev_runs for t in r.hof]).head(15)